In [1]:
import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split, StratifiedKFold,GridSearchCV
from sklearn.metrics import mean_squared_error, make_scorer, r2_score, confusion_matrix
import numpy as np
from sklearn.decomposition import PCA
from sklearn.inspection import permutation_importance
from sklearn.feature_selection import RFECV
from sklearn.model_selection import KFold
from sklearn.metrics import precision_score, recall_score

In [2]:
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

In [3]:
df = pd.read_excel('/content/all_data_0710.xlsx')


# Clean Data
df = df.drop(['Cause of Accident', '事发水域路况'], axis=1)
df=df.drop(['fatality','损伤位置','损伤种类'],axis=1)
df=df.dropna(axis=0)


In [4]:
# Create target variable
y = df["accident level"].astype(str)

# Create target to label mapping
y_encoded, y_labels = pd.factorize(y)
target_mapping = dict(zip(range(len(y_labels)), y_labels))

# Drop the original target column
df=df.drop(["accident level"],axis=1)

# One-hot encode the remaining features
df_encoded = pd.get_dummies((df), drop_first=True)

In [5]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(df_encoded, y_encoded, test_size=0.3, random_state=42)


labels_in_data = sorted(np.unique(y_test))

# parameter thoice and models
models = {
    'Gradient Boosting': (GradientBoostingClassifier(), {
        'clf__n_estimators': [100, 200],
        'clf__max_depth': [3, 5],
    }),
    'Random Forest': (RandomForestClassifier(), {
        'clf__n_estimators': [100, 200],
        'clf__max_depth': [None, 10],
    }),
    'SVM': (SVC(), {
        'clf__C': [1, 10],
        'clf__kernel': ['linear', 'rbf'],
    }),
    'KNN': (KNeighborsClassifier(), {
        'clf__n_neighbors': [3, 5, 7],
    }),
}

# save results
results = {}

# Train each model with GridSearchCV
for name, (model, param_grid) in models.items():
    pipe = Pipeline([
        ('clf', model)
    ])
    grid = GridSearchCV(pipe, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
    grid.fit(X_train, y_train)

    y_pred = grid.best_estimator_.predict(X_test)
    results[name] = {
        'Best Parameters': grid.best_params_,
        'Test Accuracy': grid.best_estimator_.score(X_test, y_test),
        'Classification Report': classification_report(
    y_test, y_pred,
    labels=labels_in_data,
    target_names = [target_mapping[i] for i in labels_in_data if i in target_mapping]
)
    }

# Print results
for model_name, metrics in results.items():
    print(f"\n=== {model_name} ===")
    print("Best Parameters:", metrics['Best Parameters'])
    print("Test Accuracy:", metrics['Test Accuracy'])
    print("Classification Report:\n", metrics['Classification Report'])

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



=== Gradient Boosting ===
Best Parameters: {'clf__max_depth': 3, 'clf__n_estimators': 100}
Test Accuracy: 0.7598425196850394
Classification Report:
               precision    recall  f1-score   support

          一般       0.82      0.91      0.86       198
          较大       0.41      0.24      0.30        50
          重大       0.00      0.00      0.00         6

    accuracy                           0.76       254
   macro avg       0.41      0.38      0.39       254
weighted avg       0.72      0.76      0.73       254


=== Random Forest ===
Best Parameters: {'clf__max_depth': None, 'clf__n_estimators': 200}
Test Accuracy: 0.8149606299212598
Classification Report:
               precision    recall  f1-score   support

          一般       0.82      0.98      0.90       198
          较大       0.71      0.24      0.36        50
          重大       0.00      0.00      0.00         6

    accuracy                           0.81       254
   macro avg       0.51      0.41      0.42     

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
model = GradientBoostingClassifier()
model.fit(X_train,y_train)  # y_encoded is from your pd.factorize

# Get feature importances
importances = model.feature_importances_
feature_names = X_train.columns

# Create a DataFrame for easy viewing
feat_importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values(by='importance', ascending=False)

print(feat_importance_df)

                     feature  importance
22       作业情况_Normal service    0.114745
67                    环境恶劣_是    0.090191
44  ship type_fishing vessel    0.083012
54                   船体材料_钢质    0.063974
36                  能见度等级_良好    0.042827
..                       ...         ...
26           作业情况_Stationary    0.000977
7             发生地种类_Open sea    0.000802
4       发生地种类_Coastal waters    0.000576
24      作业情况_Special Service    0.000317
51              ship type_未知    0.000000

[68 rows x 2 columns]
